In [1]:
#Reading the Data files
import pandas as pd
import numpy as np
import statsmodels.api as sm
import seaborn as sns
import matplotlib.pyplot as plt

analysis = pd.read_csv('/Users/chintanjikkar/Desktop/PACE/Fall 25/Predictive Analytics/Assignments/PAC/Data/analysis_data.csv')
scoring = pd.read_csv('/Users/chintanjikkar/Desktop/PACE/Fall 25/Predictive Analytics/Assignments/PAC/Data/scoring_data.csv')

In [2]:
#Importing Libraries and functions to be used 
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from statsmodels.stats.outliers_influence import variance_inflation_factor

In [3]:
#All column names for ref
analysis.columns, scoring.columns

(Index(['customer_id', 'age', 'gender', 'marital_status', 'education_level',
        'region', 'employment_status', 'owns_home', 'has_auto_loan',
        'annual_income', 'credit_score', 'credit_limit', 'tenure', 'card_type',
        'num_transactions', 'avg_transaction_value', 'online_shopping_freq',
        'reward_points_balance', 'travel_frequency', 'utility_payment_count',
        'num_children', 'num_credit_cards', 'monthly_spend'],
       dtype='object'),
 Index(['customer_id', 'age', 'gender', 'marital_status', 'education_level',
        'region', 'employment_status', 'owns_home', 'has_auto_loan',
        'annual_income', 'credit_score', 'credit_limit', 'tenure', 'card_type',
        'num_transactions', 'avg_transaction_value', 'online_shopping_freq',
        'reward_points_balance', 'travel_frequency', 'utility_payment_count',
        'num_children', 'num_credit_cards'],
       dtype='object'))

In [4]:
#Need for dummies check
analysis.dtypes, scoring.dtypes

(customer_id                int64
 age                        int64
 gender                    object
 marital_status            object
 education_level           object
 region                    object
 employment_status         object
 owns_home                  int64
 has_auto_loan              int64
 annual_income            float64
 credit_score             float64
 credit_limit             float64
 tenure                     int64
 card_type                 object
 num_transactions           int64
 avg_transaction_value    float64
 online_shopping_freq     float64
 reward_points_balance    float64
 travel_frequency           int64
 utility_payment_count    float64
 num_children               int64
 num_credit_cards           int64
 monthly_spend            float64
 dtype: object,
 customer_id                int64
 age                        int64
 gender                    object
 marital_status            object
 education_level           object
 region                    objec

In [5]:
#NA Variables
print(analysis.isnull().sum().sort_values(ascending=False).head(10))
print(scoring.isnull().sum().sort_values(ascending=False).head(10))

online_shopping_freq     2008
education_level          1199
utility_payment_count     798
customer_id                 0
tenure                      0
num_credit_cards            0
num_children                0
travel_frequency            0
reward_points_balance       0
avg_transaction_value       0
dtype: int64
online_shopping_freq     492
education_level          301
utility_payment_count    202
customer_id                0
tenure                     0
num_children               0
travel_frequency           0
reward_points_balance      0
avg_transaction_value      0
num_transactions           0
dtype: int64


In [6]:
for col in ['online_shopping_freq', 'utility_payment_count', 'education_level']:
    flag = f'{col}_was_na'
    analysis[flag] = analysis[col].isna().astype(int)
    scoring[flag]  = scoring[col].isna().astype(int)

In [7]:
#Numeric columns - fill with median 
online_med  = analysis['online_shopping_freq'].median()
utility_med = analysis['utility_payment_count'].median()

analysis['online_shopping_freq']  = analysis['online_shopping_freq'].fillna(online_med)
analysis['utility_payment_count'] = analysis['utility_payment_count'].fillna(utility_med)

scoring['online_shopping_freq']   = scoring['online_shopping_freq'].fillna(online_med)
scoring['utility_payment_count']  = scoring['utility_payment_count'].fillna(utility_med)

#Categorical column - fill with mode
edu_mode = analysis['education_level'].mode(dropna=True)
edu_fill = edu_mode.iloc[0] if len(edu_mode) else 'Unknown'

analysis['education_level'] = analysis['education_level'].fillna(edu_fill)
scoring['education_level']  = scoring['education_level'].fillna(edu_fill)


In [8]:
#Recheck
print(analysis.isnull().sum().sort_values(ascending=False).head(10))
print(scoring.isnull().sum().sort_values(ascending=False).head(10))

customer_id                     0
age                             0
utility_payment_count_was_na    0
online_shopping_freq_was_na     0
monthly_spend                   0
num_credit_cards                0
num_children                    0
utility_payment_count           0
travel_frequency                0
reward_points_balance           0
dtype: int64
customer_id                     0
card_type                       0
utility_payment_count_was_na    0
online_shopping_freq_was_na     0
num_credit_cards                0
num_children                    0
utility_payment_count           0
travel_frequency                0
reward_points_balance           0
online_shopping_freq            0
dtype: int64


In [9]:
#Creating Dummies

cat_features = [
    'gender', 'marital_status', 'education_level',
    'region', 'employment_status', 'card_type'
]

analysis_dum = pd.get_dummies(
    analysis,
    columns=[c for c in cat_features if c in analysis.columns],
    drop_first=True,
    prefix_sep='_'
)

scoring_dum = pd.get_dummies(
    scoring,
    columns=[c for c in cat_features if c in scoring.columns],
    drop_first=True,
    prefix_sep='_'
)

scoring_dum = scoring_dum.reindex(columns=analysis_dum.columns, fill_value=0)

In [10]:
#Bool Conversion for safety
bool_cols_analysis = analysis_dum.select_dtypes(include=['bool']).columns
bool_cols_scoring  = scoring_dum.select_dtypes(include=['bool']).columns

analysis_dum[bool_cols_analysis] = analysis_dum[bool_cols_analysis].astype(int)
scoring_dum[bool_cols_scoring]   = scoring_dum[bool_cols_scoring].astype(int)

In [11]:
#Sanity checks
print("Shape after dummy creation:")
print(f"Analysis: {analysis_dum.shape}")
print(f"Scoring : {scoring_dum.shape}")

print("\nSample columns after encoding:")
print(analysis_dum.columns[:10])

print("\nRemaining object columns (should be 0):")
print(analysis_dum.select_dtypes(include=['object']).columns)

Shape after dummy creation:
Analysis: (40000, 32)
Scoring : (10000, 32)

Sample columns after encoding:
Index(['customer_id', 'age', 'owns_home', 'has_auto_loan', 'annual_income',
       'credit_score', 'credit_limit', 'tenure', 'num_transactions',
       'avg_transaction_value'],
      dtype='object')

Remaining object columns (should be 0):
Index([], dtype='object')


In [12]:
train = analysis_dum.sample(frac=0.7, random_state=1031)
score = analysis_dum.drop(labels=train.index)

In [13]:
X_train = train.drop(columns=['monthly_spend'], errors='ignore')
y_train = train['monthly_spend']

X_score = score.drop(columns=['monthly_spend'], errors='ignore')
y_score = score['monthly_spend']

In [14]:
#SC
print("Training set shape :", X_train.shape)
print("Scoring  set shape :", X_score.shape)
print("Target variable OK :", y_train.shape, y_score.shape)

Training set shape : (28000, 31)
Scoring  set shape : (12000, 31)
Target variable OK : (28000,) (12000,)


In [15]:
feature_cols = [c for c in analysis_dum.columns if c != 'monthly_spend']
X_train = train[feature_cols].copy()
y_train = train['monthly_spend'].copy()

X_score = score[feature_cols].copy()
y_score = score['monthly_spend'].copy()


In [32]:
sns.pairplot(train)
plt.show()

In [33]:
lin_reg1 = LinearRegression()
lin_reg1.fit(X_train, y_train)

,fit_intercept,True
,copy_X,True
,tol,1e-06
,n_jobs,None
,positive,False


In [34]:
pred_tr = lin_reg1.predict(X_train)

rmse_tr = np.sqrt(mean_squared_error(y_train, pred_tr))
mae_tr  = mean_absolute_error(y_train, pred_tr)
r2_tr   = r2_score(y_train, pred_tr)

print(f"Reg1 (All features except target)")
print(f"Train | R2={r2_tr:.4f}  RMSE={rmse_tr:.3f}  MAE={mae_tr:.3f}")

Reg1 (All features except target)
Train | R2=0.7717  RMSE=257.653  MAE=204.109


In [ ]:
###DROPPED FOR NOW####
##BUILDING ON OLS THEN TRANSITION TO LINEAR ONLY ON SIGNIFICANT ONES##